<a href="https://colab.research.google.com/github/Frank1to/Trabajo-Bot/blob/main/bot_rag_sales_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Bot RAG — sales_data_sample.csv + Mistral API

**Stack:** Mistral API · LangChain · FAISS · Embeddings multilingüe

### Antes de empezar:
1. Obtené tu API Key gratis en [console.mistral.ai](https://console.mistral.ai) → API Keys → Create new key
2. Pegala en la Celda 2
3. Ejecutá las celdas en orden

> ✅ No necesitás GPU.

## Celda 1 — Instalación

In [ ]:
!pip install -q langchain langchain-community langchain-mistralai langchain-huggingface
!pip install -q faiss-cpu sentence-transformers mistralai pandas
print('✅ Listo')

## Celda 2 — API Key de Mistral

In [ ]:
import os
MISTRAL_API_KEY = 'u6XWGt43xByqfVH3PiyePa6ppJOTPZSB'  # <-- reemplazá esto
os.environ['MISTRAL_API_KEY'] = MISTRAL_API_KEY
print('✅ API Key configurada')

## Celda 3 — Subir y cargar el CSV

In [ ]:
import pandas as pd
import io
from google.colab import files

print('Subí el archivo sales_data_sample.csv:')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]), encoding='latin1')

# Normalizar columnas
df.columns = df.columns.str.lower().str.strip()
df['orderdate'] = pd.to_datetime(df['orderdate'], format='%Y-%m-%d')
df['territory'] = df['territory'].fillna('NA')
df['state'] = df['state'].fillna('N/A')
df['postalcode'] = df['postalcode'].fillna('Unknown').astype(str)

print(f'✅ Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'   Período: {df["orderdate"].min().year} - {df["orderdate"].max().year}')
print(f'   Productos: {", ".join(df["productline"].unique())}')
df.head(3)

## Celda 4 — Construir el Corpus para RAG

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Convertir cada fila en un documento de texto estructurado
docs_csv = []
for _, fila in df.iterrows():
    texto = (
        f"Pedido {fila['ordernumber']} | "
        f"Cliente: {fila['customername']} | "
        f"Producto: {fila['productline']} ({fila['productcode']}) | "
        f"Cantidad: {fila['quantityordered']} | "
        f"Precio unitario: ${fila['priceeach']:.2f} | "
        f"Total venta: ${fila['sales']:.2f} | "
        f"Estado: {fila['status']} | "
        f"Fecha: {fila['orderdate'].strftime('%Y-%m-%d')} | "
        f"Pais: {fila['country']} | "
        f"Territorio: {fila['territory']} | "
        f"Deal size: {fila['dealsize']}"
    )
    docs_csv.append(Document(
        page_content=texto,
        metadata={
            'fuente': 'sales_data_sample.csv',
            'ordernumber': str(fila['ordernumber']),
            'pais': str(fila['country']),
            'producto': str(fila['productline'])
        }
    ))

# Contexto de negocio (resumen del dataset)
contexto_negocio = [
    'El dataset tiene 2,823 registros de ventas de modelos a escala entre 2003 y 2005.',
    'Lineas de productos: Classic Cars, Vintage Cars, Motorcycles, Trucks and Buses, Planes, Ships, Trains.',
    'Classic Cars es la linea con mas ventas: $3,919,615. Trains es la de menos: $226,243.',
    'USA lidera las ventas con $3,627,982, seguido de Spain ($1,215,686) y France ($1,110,916).',
    'Territorios: NA (Norteamerica), EMEA (Europa/Medio Oriente/Africa), APAC (Asia-Pacifico), Japan.',
    'El total global de ventas es $10,032,628.85 con un promedio de $3,553.89 por pedido.',
    'Deal sizes: Small (1,282 pedidos), Medium (1,384 pedidos), Large (157 pedidos).',
    'Estados de pedidos: Shipped (2,617), Cancelled (60), Resolved (47), On Hold (44), In Process (41), Disputed (14).',
]
docs_contexto = [Document(page_content=t, metadata={'fuente': 'contexto'}) for t in contexto_negocio]

splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=60)
todos_los_docs = splitter.split_documents(docs_csv + docs_contexto)

print(f'✅ Corpus listo: {len(todos_los_docs):,} chunks')


## Celda 5 — Embeddings + Índice FAISS

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

print('⏳ Generando embeddings (1-2 min)...')
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
)
vectorstore = FAISS.from_documents(todos_los_docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 8})
vectorstore.save_local('faiss_sales')
print('✅ Índice FAISS creado')

## Celda 6 — Conectar Mistral

In [ ]:
from langchain_mistralai import ChatMistralAI

llm = ChatMistralAI(
    model='mistral-small-latest',
    api_key=MISTRAL_API_KEY,
    temperature=0.1,
    max_tokens=1024
)
test = llm.invoke('Respondé en una linea: estas listo?')
print('✅ Mistral conectado:', test.content)

## Celda 7 — Función principal del bot

In [ ]:
import json
from langchain_core.messages import HumanMessage, SystemMessage

SYSTEM_PROMPT = (
    "Eres un asistente experto en analisis de ventas de modelos a escala (2003-2005). "
    "Responde siempre en espanol. Se preciso con los numeros. "
    "Si te dan estadisticas calculadas, usalas directamente. "
    "Si no tenes suficiente informacion, decilo."
)

def calcular_stats(pregunta):
    p = pregunta.lower()
    stats = {}
    if any(w in p for w in ['total', 'suma']):
        stats['total_ventas'] = f"${df['sales'].sum():,.2f}"
    if any(w in p for w in ['promedio', 'media']):
        stats['promedio_venta'] = f"${df['sales'].mean():,.2f}"
    if any(w in p for w in ['maximo', 'mayor venta', 'mas alto']):
        idx = df['sales'].idxmax()
        stats['venta_maxima'] = f"${df['sales'].max():,.2f} (pedido {df.loc[idx,'ordernumber']}, {df.loc[idx,'customername']})"
    if any(w in p for w in ['producto', 'linea']):
        top = df.groupby('productline')['sales'].sum().sort_values(ascending=False)
        stats['ventas_por_producto'] = {k: f'${v:,.2f}' for k, v in top.items()}
    if any(w in p for w in ['pais', 'country']):
        top = df.groupby('country')['sales'].sum().sort_values(ascending=False).head(5)
        stats['top5_paises'] = {k: f'${v:,.2f}' for k, v in top.items()}
    if any(w in p for w in ['estado', 'status', 'cancelad', 'shipped']):
        stats['pedidos_por_estado'] = df['status'].value_counts().to_dict()
    if any(w in p for w in ['año', 'anio', '2003', '2004', '2005']):
        top = df.groupby('year_id')['sales'].sum()
        stats['ventas_por_año'] = {int(k): f'${v:,.2f}' for k, v in top.items()}
    if any(w in p for w in ['deal', 'tamano', 'small', 'medium', 'large']):
        stats['pedidos_por_dealsize'] = df['dealsize'].value_counts().to_dict()
    if any(w in p for w in ['cuantos', 'registros', 'filas', 'pedidos']):
        stats['total_registros'] = f'{len(df):,}'
    if any(w in p for w in ['territorio', 'emea', 'apac', 'japan']):
        top = df.groupby('territory')['sales'].sum().sort_values(ascending=False)
        stats['ventas_por_territorio'] = {k: f'${v:,.2f}' for k, v in top.items()}
    if any(w in p for w in ['cliente', 'customer', 'top']):
        top = df.groupby('customername')['sales'].sum().sort_values(ascending=False).head(5)
        stats['top5_clientes'] = {k: f'${v:,.2f}' for k, v in top.items()}
    return stats

def responder(pregunta):
    stats = calcular_stats(pregunta)
    docs = retriever.invoke(pregunta)
    contexto_rag = '\n'.join([d.page_content for d in docs[:5]])
    contexto = ''
    if stats:
        contexto += f'Estadisticas calculadas:\n{json.dumps(stats, ensure_ascii=False, indent=2)}\n\n'
    contexto += f'Registros relevantes del dataset:\n{contexto_rag}'
    mensajes = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f'{contexto}\n\nPregunta: {pregunta}')
    ]
    return llm.invoke(mensajes).content

print('✅ Bot listo')

## Celda 8 — Chat interactivo

Ejemplos de preguntas:
- *Cual es el total de ventas?*
- *Que linea de producto vende mas?*
- *Cuantos pedidos fueron cancelados?*
- *Cuales son los 5 mejores clientes?*
- *Como fueron las ventas por año?*

In [ ]:
print('=' * 50)
print('BOT DE VENTAS — sales_data_sample + Mistral')
print('Escribi tu pregunta o "salir" para terminar')
print('=' * 50)

while True:
    pregunta = input('\nVos: ').strip()
    if pregunta.lower() in ['salir', 'exit', 'q']:
        print('Hasta luego!')
        break
    if not pregunta:
        continue
    try:
        print('Pensando...', end='\r')
        print(f'Mistral: {responder(pregunta)}')
    except Exception as e:
        print(f'Error: {e}')

## Celda 9 — Pruebas automaticas

In [ ]:
preguntas = [
    'Cuantos registros tiene el dataset?',
    'Cual es el total de ventas?',
    'Que linea de producto genera mas ingresos?',
    'Cuales son los 5 paises con mas ventas?',
    'Cuantos pedidos fueron cancelados?',
    'Como fueron las ventas por año?',
    'Cuales son los 5 mejores clientes?',
    'Que territorio tiene mas ventas?',
]
for p in preguntas:
    print(f'\nPregunta: {p}')
    print(f'Respuesta: {responder(p)}')
    print('-' * 45)